# Compute SCimilarity embeddings

In [ ]:
!pip install git+https://github.com/Genentech/scimilarity.git@main

In [1]:
import os
from os.path import join

import scanpy as sc
import tqdm

from scimilarity.utils import lognorm_counts, align_dataset
from scimilarity import CellAnnotation

In [2]:
DATA_PATH = "/mnt/dssfs02/dataset-similarity/preprocessed"
CACHE_DIR = "/mnt/dssfs02/dataset-similarity/cache"

MODEL_PATH = "/mnt/dssfs02/scimilarity_ckpts"

In [3]:
ca = CellAnnotation(model_path=MODEL_PATH)

In [4]:
for file in tqdm.tqdm(sorted(os.listdir(DATA_PATH))):
    adata_original = sc.read_h5ad(join(DATA_PATH, file))
    adata = adata_original.copy()
    adata.layers["counts"] = adata.X
    adata.var = adata.var.set_index("feature_name")
    adata = align_dataset(adata, ca.gene_order)
    adata = lognorm_counts(adata)

    x_embed = ca.get_embeddings(adata.X)
    adata_original.obsm["X_scimilarity"] = x_embed
    adata_original.write_h5ad(join(DATA_PATH, file))


100%|██████████| 18/18 [1:10:33<00:00, 235.19s/it]
